# Datadog Remote MCP Server as an AgentCore Gateway Target (OAuth 2.1 3LO)

This tutorial connects the **Datadog Remote MCP Server** to **Amazon Bedrock AgentCore Gateway**
as an MCP target, so that agents can discover and invoke Datadog observability tools
(logs, metrics, traces, monitors, incidents, LLM Observability, security signals, and more)
through a single, governed interface.

Unlike a static OAuth app, the Datadog MCP Server issues client credentials via **Dynamic Client
Registration (DCR)** and supports **OAuth 2.1 3LO (Authorization Code + PKCE) only** — which means
every tool call is executed under the *authorized user's* Datadog RBAC, Data Access Controls, and
audit identity. Per-user attribution is preserved end-to-end.

| Information | Details |
|-------------|---------|
| Tutorial type | Interactive |
| AgentCore components | Gateway, Identity |
| Agentic framework | Strands Agents |
| Gateway target type | MCP server |
| Inbound auth IdP | Amazon Cognito (JWT) |
| Outbound auth | OAuth 2.1 3LO (Authorization Code + PKCE) via DCR |
| LLM model | Anthropic Claude Sonnet 4 |
| SDK | boto3 |
| Vertical | Observability |
| Complexity | Intermediate |

## Architecture

```
[Strands Agent]
   |  Cognito JWT  +  MCP-Protocol-Version: 2025-11-25
   v
[AgentCore Gateway]  (JWT inbound, MCP 2025-11-25)
   |  per-user Datadog access token (from AgentCore Token Vault)
   v
[Datadog Remote MCP Server]
   |  enforces the authorized user's RBAC / Data Access Controls / Log Restriction Queries
   v
[Datadog data: logs, metrics, traces, monitors, incidents, LLMObs, ...]
```

## Prerequisites

- An AWS account with access to Amazon Bedrock AgentCore (Gateway + Identity).
- Permissions to create Cognito user pools, AgentCore gateways, targets, and OAuth credential providers.
- A **Datadog account** and the ability to complete an interactive Datadog login in your browser
  during the per-user authorization step.
- Model access to **Anthropic Claude Sonnet 4** in Amazon Bedrock (for the agent demo).
- Python 3.10+.

Install dependencies:

In [ ]:
!pip install -r requirements.txt -q

## Datadog Remote MCP Server — key facts

These determine the exact configuration below. (Sources: Datadog public docs at
`https://docs.datadoghq.com/bits_ai/mcp_server/setup/` and the official repo
`https://github.com/datadog-labs/mcp-server`.)

- **MCP endpoint (US1):** `https://mcp.datadoghq.com/api/unstable/mcp-server/mcp`
  - For other sites, swap the host: `mcp.datadoghq.eu` (EU), `us3`/`us5`/`ap1`/`ap2` variants.
- **Auth:** OAuth 2.1 **3LO only** (Authorization Code + PKCE). `client_credentials` (2LO) is **not** supported.
- **No OAuth app portal.** Credentials are obtained only via **Dynamic Client Registration (DCR)**:
  - Register endpoint: `https://mcp.datadoghq.com/api/unstable/mcp-server/register`
  - Authorize endpoint: `https://mcp.datadoghq.com/api/unstable/mcp-server/authorize`
  - Token endpoint:    `https://mcp.datadoghq.com/api/unstable/mcp-server/token`
- **Discovery:** Datadog serves OAuth 2.0 discovery (`.well-known/oauth-authorization-server`), **not OIDC**.
  AgentCore's "Discovery URL" mode requires OIDC, so we configure the endpoints **manually**.
- **Toolset scoping:** append `?toolsets=core,llmobs,...` to the MCP endpoint (default is `core`).
- **Permissions:** the Datadog user needs `mcp_read` (and `mcp_write` for write tools) plus the
  relevant resource permissions (e.g. Monitors Read).

> **AgentCore requirements for 3LO (all four must hold):**
> 1. Gateway MCP **Supported version `2025-11-25` or later** (the console default `2025-03-26` hides 3LO).
> 2. Inbound Auth = **JWT (Cognito)** — IAM/SigV4 rejects 3LO targets.
> 3. Outbound Auth grant = **Authorization code grant (3LO)** at the target level.
> 4. A **Return URL** is required for 3LO targets (any URL is fine for testing).

In [ ]:
import boto3, json, requests

# ---- Configuration -------------------------------------------------------
REGION = "us-west-2"  # your AgentCore region

# Datadog site endpoints (US1 shown; change host for other sites)
DD_BASE = "https://mcp.datadoghq.com/api/unstable/mcp-server"
DD_MCP_ENDPOINT  = f"{DD_BASE}/mcp"
DD_MCP_ENDPOINT += "?toolsets=core,llmobs"   # scope the tools exposed through the gateway
DD_REGISTER_URL  = f"{DD_BASE}/register"
DD_AUTHORIZE_URL = f"{DD_BASE}/authorize"
DD_TOKEN_URL     = f"{DD_BASE}/token"
DD_ISSUER        = "https://mcp.datadoghq.com"

GATEWAY_NAME = "datadog-mcp-gateway"
TARGET_NAME  = "datadog-mcp"
MCP_PROTOCOL_VERSION = "2025-11-25"   # MUST be >= this for 3LO

import os
os.environ.setdefault("AWS_DEFAULT_REGION", REGION)  # utils.py reads the session region

# GATEWAY_ROLE_ARN is set in the next cell by creating the IAM role the Gateway assumes.

# AgentCore control-plane + identity clients
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
identity  = boto3.client("bedrock-agentcore-control", region_name=REGION)  # identity APIs live on the control plane

## Step 0 — Create the IAM role the Gateway will assume

AgentCore Gateway needs an IAM role (trust policy allowing `bedrock-agentcore.amazonaws.com` to
assume it, plus permissions for `bedrock-agentcore:*`, `bedrock:*`, `agent-credential-provider:*`,
`iam:PassRole`, `secretsmanager:GetSecretValue`, `lambda:InvokeFunction`).

This repo ships a helper (`gateway/utils.py`) that creates exactly that role — the same one the
Dynatrace sample uses. Your AWS identity must be allowed to create IAM roles (`iam:CreateRole`,
`iam:PutRolePolicy`). If it isn't, have an admin create the role with the policy above and set
`GATEWAY_ROLE_ARN` manually instead of running this cell.

In [ ]:
# Import the shared helper from gateway/utils.py (one level up from this folder)
import sys, os
utils_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, utils_dir)
import utils

gateway_role = utils.create_agentcore_gateway_role("datadog-mcpgateway")
GATEWAY_ROLE_ARN = gateway_role["Role"]["Arn"]
print("Gateway role ARN:", GATEWAY_ROLE_ARN)

## Step 1 — Create the OAuth credential provider (with placeholder credentials)

DCR has a chicken-and-egg problem: you need a **redirect/callback URI** to register the client,
but that callback URI only exists *after* AgentCore creates the credential provider. So we first
create the provider with `placeholder` client id/secret, read back its generated callback URL,
perform DCR, then swap in the real credentials (Step 3).

We use **manual** endpoint configuration (not Discovery URL) because Datadog is OAuth 2.0, not OIDC.

> **Two deviations from the Dynatrace sample to be aware of (verify against your SDK version):**
> 1. **Manual OAuth config, not `discoveryUrl`.** The Dynatrace sample passes
>    `oauthDiscovery: {discoveryUrl: ...}` (OIDC). Datadog has no OIDC discovery, so we provide the
>    authorization-server metadata (issuer / authorize / token endpoints) directly. If your
>    `bedrock-agentcore` SDK names this field differently, mirror the **console "Manual config"**
>    fields, which are verified to work.
> 2. **3LO outbound on the target.** The Dynatrace target call only sets `scopes`. For Datadog 3LO
>    you must select **Authorization code grant (3LO)** and provide a return URL — set below via
>    `grantType="AUTHORIZATION_CODE"` and `defaultReturnUrl`. (Field name is `defaultReturnUrl`, not
>    `returnUrl`.) If your SDK version rejects these, set the grant type and return URL on the target
>    via the **AgentCore console** (the verified path).

In [ ]:
provider_resp = identity.create_oauth2_credential_provider(
    name="datadog-mcp-oauth",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "clientId": "placeholder",
            "clientSecret": "placeholder",
            "oauthDiscovery": {
                "authorizationServerMetadata": {
                    "issuer": DD_ISSUER,
                    "authorizationEndpoint": DD_AUTHORIZE_URL,
                    "tokenEndpoint": DD_TOKEN_URL,
                    "responseTypes": ["code"],
                }
            },
        }
    },
)
provider_arn = provider_resp["credentialProviderArn"]
print("Provider ARN:", provider_arn)

# The callback URL AgentCore will use as the OAuth redirect_uri.
# (Field name may surface as 'callbackUrl' / in the provider detail — print the full
#  response if you don't see it and copy the callback URL containing a UUID.)
print(json.dumps(provider_resp, indent=2, default=str))
CALLBACK_URL = provider_resp.get("callbackUrl")  # TODO: confirm field name in your SDK version
print("Callback URL:", CALLBACK_URL)

## Step 2 — Dynamic Client Registration against Datadog

Register the AgentCore callback URL with Datadog's MCP server. AgentCore's callback URI pattern is
pre-allowlisted on the Datadog side, so no per-customer approval is needed. This returns the real
`client_id` / `client_secret`.

In [ ]:
dcr_payload = {
    "client_name": "agentcore-gateway",
    "redirect_uris": [CALLBACK_URL],
    "grant_types": ["authorization_code", "refresh_token"],
    "response_types": ["code"],
    "token_endpoint_auth_method": "client_secret_basic",
}
dcr = requests.post(DD_REGISTER_URL, json=dcr_payload,
                    headers={"Content-Type": "application/json"}, timeout=30)
dcr.raise_for_status()
creds = dcr.json()
DD_CLIENT_ID = creds["client_id"]
DD_CLIENT_SECRET = creds["client_secret"]
print("Registered client_id:", DD_CLIENT_ID)

## Step 3 — Replace the placeholder credentials with the real ones

In [ ]:
identity.update_oauth2_credential_provider(
    name="datadog-mcp-oauth",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "clientId": DD_CLIENT_ID,
            "clientSecret": DD_CLIENT_SECRET,
            "oauthDiscovery": {
                "authorizationServerMetadata": {
                    "issuer": DD_ISSUER,
                    "authorizationEndpoint": DD_AUTHORIZE_URL,
                    "tokenEndpoint": DD_TOKEN_URL,
                    "responseTypes": ["code"],
                }
            },
        }
    },
)
print("Credential provider updated with DCR credentials.")

## Step 4 — Create the Gateway (MCP 2025-11-25, Cognito JWT inbound)

We use Cognito Quick Create for the inbound JWT authorizer. The **supported MCP version must be
`2025-11-25`** or the 3LO outbound option is silently unavailable.

In [ ]:
# --- Cognito user pool for inbound JWT (quick create) ---------------------
cognito = boto3.client("cognito-idp", region_name=REGION)
pool = cognito.create_user_pool(PoolName="datadog-mcp-gateway-pool")
USER_POOL_ID = pool["UserPool"]["Id"]
app_client = cognito.create_user_pool_client(
    UserPoolId=USER_POOL_ID, ClientName="datadog-mcp-gateway-client",
    GenerateSecret=False, ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
)
APP_CLIENT_ID = app_client["UserPoolClient"]["ClientId"]
DISCOVERY = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
print("User pool:", USER_POOL_ID, "| app client:", APP_CLIENT_ID)

In [ ]:
gw = agentcore.create_gateway(
    name=GATEWAY_NAME,
    roleArn=GATEWAY_ROLE_ARN,
    protocolType="MCP",
    protocolConfiguration={
        "mcp": {
            "supportedVersions": [MCP_PROTOCOL_VERSION],
            "searchType": "SEMANTIC",
        }
    },
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": DISCOVERY,
            "allowedClients": [APP_CLIENT_ID],
        }
    },
)
GATEWAY_ID = gw["gatewayId"]
GATEWAY_URL = gw["gatewayUrl"]
print("Gateway:", GATEWAY_ID, GATEWAY_URL)

## Step 5 — Create the MCP target with OAuth 3LO outbound auth

Point the target at the Datadog MCP endpoint, attach the OAuth credential provider, and set the
outbound grant to **Authorization code grant (3LO)** with a return URL.

In [ ]:
target = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=TARGET_NAME,
    targetConfiguration={
        "mcp": {"mcpServer": {"endpoint": DD_MCP_ENDPOINT}}
    },
    credentialProviderConfigurations=[{
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": provider_arn,
                "scopes": [],                      # toolset scoping is via the endpoint query param
                "grantType": "AUTHORIZATION_CODE",  # 3LO
                "defaultReturnUrl": "https://example.com",  # any URL for testing
            }
        },
    }],
)
TARGET_ID = target["targetId"]
print("Target:", TARGET_ID, "| status:", target.get("status"))

## Step 6 — Authorize the target (per-user 3LO browser flow)

3LO requires an interactive browser login: the user authenticates to **the correct Datadog org/site**
and clicks **Allow**. AgentCore then stores the user's access + refresh token in its Token Vault and
the target becomes **Ready**.

The simplest way to complete this is the **AgentCore console**: open the gateway, find the target
showing *Needs authorization*, click **Authorize**, and complete the Datadog login. Be quick — the
PAR session has a short TTL, and switching orgs mid-flow can produce `Invalid request`.

Programmatically, you can trigger the same 3LO flow with the AgentCore Identity SDK, which returns an
authorization URL to open in a browser. Adjust to your installed `bedrock-agentcore` SDK version:

In [ ]:
# Example using the bedrock-agentcore identity helper to start the 3LO flow.
# The exact helper/signature varies by SDK version; the console Authorize button does the same thing.
from bedrock_agentcore.identity.auth import requires_access_token  # noqa

# When invoked, this opens (or prints) the Datadog authorization URL. Log into the correct
# Datadog org and click Allow. The token is cached in the AgentCore Token Vault per user.
print("Open the AgentCore console -> Gateway -> Target -> Authorize, and log into Datadog.")
print("Wait for the target status to become 'Ready' before invoking.")

## Step 7 — Invoke the Gateway directly (list Datadog tools)

Every call must include the Cognito JWT and the `MCP-Protocol-Version` header. A successful
`tools/list` returns the Datadog MCP tool set (scoped to `core,llmobs` here), enforcing the
authorized user's Datadog RBAC and Data Access Controls.

In [ ]:
# Get a Cognito JWT for a test user (create/confirm a user out-of-band, then authenticate).
# Replace with your test user credentials.
auth = cognito.initiate_auth(
    ClientId=APP_CLIENT_ID, AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": "<TEST_USER>", "PASSWORD": "<TEST_PASSWORD>"},
)
JWT = auth["AuthenticationResult"]["AccessToken"]

resp = requests.post(
    GATEWAY_URL,
    headers={
        "Authorization": f"Bearer {JWT}",
        "Content-Type": "application/json",
        "MCP-Protocol-Version": MCP_PROTOCOL_VERSION,
    },
    json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}},
    timeout=60,
)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2)[:4000])

## Step 8 — Use the tools from a Strands agent

Finally, let a Strands agent (Claude Sonnet 4) discover and call the Datadog tools through the
gateway. The agent uses semantic tool search (`x_amz_bedrock_agentcore_search`) and then invokes
the relevant Datadog tool to answer an observability question.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

def dd_mcp_client():
    return streamablehttp_client(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {JWT}",
            "MCP-Protocol-Version": MCP_PROTOCOL_VERSION,
        },
    )

mcp_client = MCPClient(dd_mcp_client)
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Discovered {len(tools)} Datadog tools through the gateway")
    agent = Agent(
        model="anthropic.claude-sonnet-4-20250514-v1:0",
        tools=tools,
    )
    result = agent("What are the open P1 incidents and any recent error-rate spikes in my Datadog org?")
    print(result)

## Cleanup

In [ ]:
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# agentcore.delete_gateway(gatewayIdentifier=GATEWAY_ID)
# identity.delete_oauth2_credential_provider(name="datadog-mcp-oauth")
# cognito.delete_user_pool(UserPoolId=USER_POOL_ID)
print("Uncomment to tear down resources.")

## Choosing OAuth 3LO vs. API key

This sample uses **OAuth 3LO** so each call runs under the authorized user's Datadog identity
(per-user RBAC, Data Access Controls, and audit log via `@metadata.oauth_client.uuid`).

If you only need an agent that calls Datadog with a single shared identity (internal automation),
the simpler path is API-key auth: send `DD_API_KEY` and `DD_APPLICATION_KEY` as HTTP headers to the
MCP endpoint and skip the DCR + Cognito + 3LO steps entirely. The trade-off is no per-user RBAC or
attribution.

---

### Notes & sources
- Datadog MCP setup: https://docs.datadoghq.com/bits_ai/mcp_server/setup/
- Official repo: https://github.com/datadog-labs/mcp-server
- Some boto3 field names (e.g. the callback URL field, identity helper signatures) can vary by
  `bedrock-agentcore` SDK version — print full responses and adjust if a key differs. The console
  Authorize button performs the same 3LO flow as the programmatic helper.